In [ ]:
# Cell 1: Install Google Chrome + required Python packages

!pip install -q selenium youtube-transcript-api webdriver-manager

!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y -qq ./google-chrome-stable_current_amd64.deb

print("✅ Google Chrome and Python packages installed!")

!google-chrome --version

✅ Google Chrome and Python packages installed!
Google Chrome 151.0.7922.137 


In [ ]:
# Cell 2: Import required libraries

import os
import json
import time
import re
from typing import List, Dict, Optional, Any
from datetime import datetime

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service

from webdriver_manager.chrome import ChromeDriverManager

from youtube_transcript_api import (
    YouTubeTranscriptApi,
    NoTranscriptFound,
    TranscriptsDisabled,
    VideoUnavailable
)

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


In [ ]:
# Cell 3: Browser diagnostics

import shutil
import subprocess

print("=" * 70)
print("🌐 BROWSER DIAGNOSTICS")
print("=" * 70)

chrome_path = shutil.which("google-chrome")
chromedriver_path = shutil.which("chromedriver")

print(f"Google Chrome: {chrome_path}")
print(f"ChromeDriver: {chromedriver_path}")

print("\n=== Versions ===")

try:
    result = subprocess.run(
        ["google-chrome", "--version"],
        capture_output=True,
        text=True
    )
    print("Chrome:", result.stdout.strip())
except Exception as e:
    print("Chrome version error:", e)

try:
    driver_path = ChromeDriverManager().install()

    result = subprocess.run(
        [driver_path, "--version"],
        capture_output=True,
        text=True
    )

    print("ChromeDriver:", result.stdout.strip())
except Exception as e:
    print("ChromeDriver version error:", e)

print("=" * 70)

🌐 BROWSER DIAGNOSTICS
Google Chrome: /usr/bin/google-chrome
ChromeDriver: /usr/bin/chromedriver

=== Versions ===
Chrome: Google Chrome 151.0.7922.137
ChromeDriver: ChromeDriver 151.0.7922.138 (41fa82442390a4d4456c78f2d69a832d5720cb27-refs/branch-heads/7922@{#2891})


In [ ]:
# Cell 4: YouTube Transcript Agent

class YouTubeTranscriptAgent:
    """
    Agent that searches YouTube using Selenium
    and fetches transcripts for the top results.
    """

    def __init__(self, languages: List[str] = None, headless: bool = True):

        if languages is None:
            languages = ["en"]

        self.languages = languages
        self.headless = headless
        self.ytt_api = YouTubeTranscriptApi()
        self.results = {}
        self.driver = None

        print(f"✅ Agent initialized with languages: {languages}")

    # ---------------------------------------------------------
    # Initialize Chrome
    # ---------------------------------------------------------

    def _init_driver(self):

        print("🌐 Initializing Google Chrome...")

        chrome_options = Options()

        if self.headless:
            chrome_options.add_argument("--headless=new")

        chrome_options.add_argument("--no-sandbox")
        chrome_options.add_argument("--disable-dev-shm-usage")
        chrome_options.add_argument("--disable-gpu")
        chrome_options.add_argument("--disable-extensions")
        chrome_options.add_argument("--disable-setuid-sandbox")
        chrome_options.add_argument("--window-size=1920,1080")

        # Important for Colab
        chrome_options.binary_location = "/usr/bin/google-chrome"

        try:

            driver_path = ChromeDriverManager().install()

            print(f"ChromeDriver path: {driver_path}")

            service = Service(driver_path)

            self.driver = webdriver.Chrome(
                service=service,
                options=chrome_options
            )

            print("✅ Chrome driver initialized successfully")

            return self.driver

        except Exception as e:

            print(f"❌ Failed to initialize Chrome driver:")
            print(e)

            raise

    # ---------------------------------------------------------
    # Search YouTube
    # ---------------------------------------------------------

    def search_youtube(
        self,
        person: str,
        topic: str = None,
        max_results: int = 3
    ) -> List[Dict]:

        if topic:
            query = f"{person} {topic}"
        else:
            query = f"{person} interview"

        print(f"\n🔍 Searching YouTube for: '{query}'")

        try:

            if not self.driver:
                self._init_driver()

            print("  Navigating to YouTube...")

            self.driver.get(
                "https://www.youtube.com"
            )

            time.sleep(3)

            print("  Entering search query...")

            search_box = WebDriverWait(
                self.driver,
                15
            ).until(
                EC.presence_of_element_located(
                    (By.NAME, "search_query")
                )
            )

            search_box.clear()
            search_box.send_keys(query)
            search_box.send_keys(Keys.RETURN)

            print("  Waiting for results...")

            time.sleep(5)

            print("  Extracting video information...")

            WebDriverWait(
                self.driver,
                15
            ).until(
                EC.presence_of_element_located(
                    (By.ID, "video-title")
                )
            )

            video_elements = self.driver.find_elements(
                By.ID,
                "video-title"
            )

            videos = []

            for element in video_elements:

                if len(videos) >= max_results:
                    break

                try:

                    href = element.get_attribute("href")

                    if not href:
                        continue

                    video_id = self._extract_video_id(href)

                    if not video_id:
                        continue

                    title = (
                        element.get_attribute("title")
                        or element.text
                    )

                    if not title:
                        continue

                    video_info = {
                        "video_id": video_id,
                        "title": title,
                        "url": href,
                        "channel": "Unknown",
                        "search_query": query
                    }

                    videos.append(video_info)

                    print(
                        f"  Found: {title[:70]}"
                    )

                except Exception as e:

                    print(
                        f"  ⚠️ Error extracting video: {e}"
                    )

                    continue

            print(
                f"✅ Found {len(videos)} videos"
            )

            return videos

        except Exception as e:

            print(f"❌ YouTube search failed:")
            print(e)

            return []

    # ---------------------------------------------------------
    # Extract YouTube video ID
    # ---------------------------------------------------------

    def _extract_video_id(
        self,
        url: str
    ) -> Optional[str]:

        patterns = [

            r"(?:youtube\.com/watch\?v=)([\w-]+)",

            r"(?:youtu\.be/)([\w-]+)",

            r"(?:youtube\.com/embed/)([\w-]+)",

            r"(?:youtube\.com/shorts/)([\w-]+)"

        ]

        for pattern in patterns:

            match = re.search(
                pattern,
                url
            )

            if match:
                return match.group(1)

        return None

    # ---------------------------------------------------------
    # Fetch transcript
    # ---------------------------------------------------------

    def fetch_transcript(
        self,
        video_info: Dict
    ) -> Optional[Dict]:

        video_id = video_info["video_id"]

        title = video_info.get(
            "title",
            video_id
        )

        try:

            print(
                f"  📝 Fetching transcript for: "
                f"{title[:60]}..."
            )

            transcript_list = self.ytt_api.list(
                video_id
            )

            try:

                transcript = (
                    transcript_list.find_transcript(
                        self.languages
                    )
                )

            except NoTranscriptFound:

                available = list(
                    transcript_list
                )

                if not available:

                    print(
                        "  ⚠️ No transcripts available"
                    )

                    return None

                transcript = available[0]

                print(
                    f"  ℹ️ Using "
                    f"{transcript.language} transcript"
                )

            fetched = transcript.fetch()

            full_text = " ".join(
                snippet.text
                for snippet in fetched
            )

            result = {

                "video_id": video_id,

                "title": video_info.get(
                    "title",
                    ""
                ),

                "url": video_info.get(
                    "url",
                    f"https://www.youtube.com/watch?v={video_id}"
                ),

                "channel": video_info.get(
                    "channel",
                    "Unknown"
                ),

                "search_query": video_info.get(
                    "search_query",
                    ""
                ),

                "language": transcript.language,

                "language_code": transcript.language_code,

                "is_generated": transcript.is_generated,

                "is_translatable": transcript.is_translatable,

                "transcript_length": len(full_text),

                "snippet_count": len(fetched),

                # Limit stored transcript size
                "full_text": full_text[:15000],

                "timestamp": datetime.now().isoformat(),

                "snippets": [

                    {
                        "text": snippet.text,
                        "start": snippet.start,
                        "duration": snippet.duration
                    }

                    for snippet in fetched[:30]

                ]

            }

            print(
                f"  ✅ Success: "
                f"{result['snippet_count']} snippets, "
                f"{result['transcript_length']:,} chars"
            )

            return result

        except TranscriptsDisabled:

            print(
                "  ❌ Transcripts disabled"
            )

        except VideoUnavailable:

            print(
                "  ❌ Video unavailable"
            )

        except NoTranscriptFound:

            print(
                f"  ❌ No transcript found "
                f"in {self.languages}"
            )

        except Exception as e:

            print(
                f"  ❌ Transcript error: {e}"
            )

        return None

    # ---------------------------------------------------------
    # Main research method
    # ---------------------------------------------------------

    def research(
        self,
        person: str,
        topic: str = None,
        max_videos: int = 3
    ) -> Dict[str, Any]:

        print("\n" + "=" * 70)
        print("🎥 YOUTUBE TRANSCRIPT RESEARCH")
        print("=" * 70)

        print(f"Person: {person}")

        if topic:
            print(f"Topic: {topic}")

        print(
            f"Languages: {', '.join(self.languages)}"
        )

        print(
            f"Headless mode: {self.headless}"
        )

        print()

        start_time = time.time()

        try:

            videos = self.search_youtube(
                person,
                topic,
                max_results=max_videos
            )

            if not videos:

                result = {

                    "person": person,

                    "topic": topic,

                    "success": False,

                    "error": "No videos found",

                    "videos_found": 0,

                    "videos_processed": 0,

                    "transcripts": []

                }

                self.results = result

                return result

            transcripts = []

            for i, video in enumerate(
                videos,
                1
            ):

                print(
                    f"\n[{i}/{len(videos)}] "
                    f"Processing video..."
                )

                transcript = (
                    self.fetch_transcript(
                        video
                    )
                )

                if transcript:

                    transcripts.append(
                        transcript
                    )

                time.sleep(1)

            elapsed_time = (
                time.time()
                - start_time
            )

            result = {

                "person": person,

                "topic": topic,

                "search_query": (
                    f"{person} {topic}"
                    if topic
                    else f"{person} interview"
                ),

                "success": (
                    len(transcripts) > 0
                ),

                "videos_found": len(videos),

                "videos_processed": len(transcripts),

                "languages_requested": self.languages,

                "languages_found": list(
                    set(
                        t["language"]
                        for t in transcripts
                    )
                ),

                "total_transcript_chars": sum(
                    t["transcript_length"]
                    for t in transcripts
                ),

                "elapsed_seconds": round(
                    elapsed_time,
                    2
                ),

                "transcripts": transcripts

            }

            self.results = result

            print("\n" + "=" * 70)
            print("📊 RESEARCH SUMMARY")
            print("=" * 70)

            print(
                f"Videos found: "
                f"{result['videos_found']}"
            )

            print(
                f"Successfully processed: "
                f"{result['videos_processed']}"
            )

            print(
                f"Languages found: "
                f"{', '.join(result['languages_found'])}"
            )

            print(
                f"Total transcript text: "
                f"{result['total_transcript_chars']:,} characters"
            )

            print(
                f"Time elapsed: "
                f"{result['elapsed_seconds']} seconds"
            )

            return result

        finally:

            self.close()

    # ---------------------------------------------------------
    # Combine transcripts
    # ---------------------------------------------------------

    def combine_transcripts(self) -> str:

        if not self.results:
            return ""

        transcripts = self.results.get(
            "transcripts",
            []
        )

        if not transcripts:
            return ""

        combined = []

        for i, transcript in enumerate(
            transcripts,
            1
        ):

            combined.append(
                "\n" + "=" * 70
            )

            combined.append(
                f"VIDEO {i}: "
                f"{transcript['title']}"
            )

            combined.append(
                "=" * 70
            )

            combined.append(
                f"Channel: "
                f"{transcript.get('channel', 'Unknown')}"
            )

            combined.append(
                f"URL: "
                f"{transcript['url']}"
            )

            combined.append(
                f"Language: "
                f"{transcript['language']}"
            )

            combined.append(
                f"Snippets: "
                f"{transcript['snippet_count']}"
            )

            combined.append(
                "\nTRANSCRIPT:\n"
                + transcript["full_text"]
            )

        return "\n".join(combined)

    # ---------------------------------------------------------
    # Save JSON
    # ---------------------------------------------------------

    def save_results(
        self,
        filename: str = None
    ) -> str:

        if not self.results:

            print(
                "⚠️ No results to save"
            )

            return ""

        if not filename:

            person = (
                self.results["person"]
                .replace(" ", "_")
                .lower()
            )

            # IMPORTANT:
            # Handles topic=None safely
            topic = (
                self.results.get("topic")
                or "general"
            )

            topic = (
                topic
                .replace(" ", "_")
                .lower()
            )

            timestamp = datetime.now().strftime(
                "%Y%m%d_%H%M%S"
            )

            filename = (
                f"youtube_"
                f"{person}_"
                f"{topic}_"
                f"{timestamp}.json"
            )

        with open(
            filename,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                self.results,
                f,
                indent=2,
                ensure_ascii=False
            )

        print(
            f"\n💾 Results saved to: "
            f"{filename}"
        )

        return filename

    # ---------------------------------------------------------
    # Save transcript TXT
    # ---------------------------------------------------------

    def save_transcript_text(
        self,
        filename: str = None
    ) -> str:

        if not self.results:

            print(
                "⚠️ No results available"
            )

            return ""

        if not self.results.get(
            "transcripts"
        ):

            print(
                "⚠️ No transcripts to save"
            )

            return ""

        if not filename:

            person = (
                self.results["person"]
                .replace(" ", "_")
                .lower()
            )

            # IMPORTANT:
            # Handles topic=None safely
            topic = (
                self.results.get("topic")
                or "general"
            )

            topic = (
                topic
                .replace(" ", "_")
                .lower()
            )

            timestamp = datetime.now().strftime(
                "%Y%m%d_%H%M%S"
            )

            filename = (
                f"transcript_"
                f"{person}_"
                f"{topic}_"
                f"{timestamp}.txt"
            )

        combined_text = (
            self.combine_transcripts()
        )

        with open(
            filename,
            "w",
            encoding="utf-8"
        ) as f:

            f.write(combined_text)

        print(
            f"💾 Transcript text saved to: "
            f"{filename}"
        )

        return filename

    # ---------------------------------------------------------
    # Close browser
    # ---------------------------------------------------------

    def close(self):

        if self.driver:

            try:
                self.driver.quit()
            except Exception:
                pass

            self.driver = None


print(
    "✅ YouTubeTranscriptAgent class defined successfully!"
)

✅ YouTubeTranscriptAgent class defined successfully!


In [ ]:
# Cell 5: Helper function for Google Colab

def colab_research(
    person: str,
    topic: str = None,
    max_videos: int = 3
):

    agent = YouTubeTranscriptAgent(
        languages=["en"],
        headless=True
    )

    results = agent.research(
        person=person,
        topic=topic,
        max_videos=max_videos
    )

    if results.get("success"):

        # Save JSON
        agent.save_results()

        # Save TXT
        agent.save_transcript_text()

        # Display sample
        if results.get("transcripts"):

            print("\n" + "=" * 70)
            print(
                "📝 SAMPLE TRANSCRIPT "
                "(first 500 characters)"
            )
            print("=" * 70)

            sample = (
                results["transcripts"][0]
                ["full_text"][:500]
            )

            print(sample + "...")

    else:

        print("\n❌ Research failed.")

        if results.get("error"):
            print(
                f"Reason: "
                f"{results['error']}"
            )

    return results


print(
    "✅ Helper function defined successfully!"
)

✅ Helper function defined successfully!


In [ ]:
# Cell 6: Run interactive research

print("=" * 70)
print("🎥 YouTube Transcript Agent for Google Colab")
print("=" * 70)

person = input(
    "Enter person name: "
).strip()

topic_input = input(
    "Enter topic (optional, press Enter to skip): "
).strip()

topic = (
    topic_input
    if topic_input
    else None
)

videos_input = input(
    "Number of videos to process (default 3): "
).strip()

max_videos = (
    int(videos_input)
    if videos_input
    else 3
)

if not person:

    raise ValueError(
        "❌ Person name cannot be empty."
    )

if max_videos < 1:

    raise ValueError(
        "❌ Number of videos must be at least 1."
    )

print("\nStarting research...\n")

results = colab_research(
    person=person,
    topic=topic,
    max_videos=max_videos
)


🎥 YouTube Transcript Agent for Google Colab
Enter person name: elon musk
Enter topic (optional, press Enter to skip): 
Number of videos to process (default 3): 

Starting research...

✅ Agent initialized with languages: ['en']

🎥 YOUTUBE TRANSCRIPT RESEARCH
Person: elon musk
Languages: en
Headless mode: True


🔍 Searching YouTube for: 'elon musk interview'
🌐 Initializing Google Chrome...
ChromeDriver path: /root/.wdm/drivers/chromedriver/linux64/151.0.7922.138/chromedriver-linux64/chromedriver
✅ Chrome driver initialized successfully
  Navigating to YouTube...
  Entering search query...
  Waiting for results...
  Extracting video information...
  Found: Elon Musk LAUGHS At Reporter Who Got EVERYTHING Wrong About Him And He
  Found: The full-length interview with Elon Musk | The Economist
  Found: Elon Musk – "In 36 months, the cheapest place to put AI will be space”
✅ Found 3 videos

[1/3] Processing video...
  📝 Fetching transcript for: Elon Musk LAUGHS At Reporter Who Got EVERYTHING 

In [ ]:
from google.colab import files

files.download("transcript_elon_musk_general_20260814_153323.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>